In [1]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install sentence-transformers

In [2]:
# Download SNAP Facebook dataset directly

import os
import zipfile

# ─── Download dataset ───
!wget -q https://snap.stanford.edu/data/facebook_large.zip

# ─── Extract to writable directory ───
extract_path = "/kaggle/working/facebook_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile("facebook_large.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset ready at:", extract_path)

# ─── Show files ───
print(os.listdir(extract_path))

Dataset ready at: /kaggle/working/facebook_data
['facebook_large']


In [3]:
import os

DATA_PATH = "/kaggle/working/facebook_data/facebook_large"

edges_path = os.path.join(DATA_PATH, "musae_facebook_edges.csv")
features_path = os.path.join(DATA_PATH, "musae_facebook_features.json")
target_path = os.path.join(DATA_PATH, "musae_facebook_target.csv")

In [4]:
import pandas as pd

edges = pd.read_csv(edges_path)

print(edges.head())
print("Total edges:", len(edges))

   id_1   id_2
0     0  18427
1     1  21708
2     1  22208
3     1  22171
4     1   6829
Total edges: 171002


In [5]:
import csv
from sklearn.preprocessing import LabelEncoder

labels_dict = {}
page_meta = {}

with open(target_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        nid = int(row["id"])
        labels_dict[nid] = row["page_type"]
        page_meta[nid] = {
            "page_name": (row.get("page_name") or "").strip(),
            "facebook_id": (row.get("facebook_id") or "").strip(),
        }

nodes = sorted(labels_dict.keys())

le = LabelEncoder()
Y = le.fit_transform([labels_dict[n] for n in nodes])

print("Classes:", len(set(Y)))

Classes: 4


In [6]:
import json
import numpy as np

with open(features_path, "r") as f:
    features_dict = json.load(f)

node_to_idx = {node: i for i, node in enumerate(nodes)}

max_feature = 0
for feats in features_dict.values():
    if feats:
        max_feature = max(max_feature, max(feats))

X = np.zeros((len(nodes), max_feature + 1))

for node_str, feats in features_dict.items():
    node = int(node_str)          # ✅ FIX: JSON keys are strings, nodes are ints
    if node in node_to_idx and feats:
        X[node_to_idx[node], feats] = 1

print("Feature shape:", X.shape)

Feature shape: (22470, 4714)


In [7]:
documents = []

for node in nodes:
    meta = page_meta[node]
    name = meta["page_name"] or "unknown"
    fb_id = meta["facebook_id"] or "n/a"
    doc = f"""
    Graph node index: {node}
    Page display name (metadata field page_name, not the prediction target): {name}
    Facebook numeric id (metadata): {fb_id}
    Facebook page in the MU SocialAds / SNAP graph; page_type label is omitted from this text.
    """
    documents.append(doc)

print(documents[:2])

['\n    Graph node index: 0\n    Page display name (metadata field page_name, not the prediction target): The Voice of China 中国好声音\n    Facebook numeric id (metadata): 145647315578475\n    Facebook page in the MU SocialAds / SNAP graph; page_type label is omitted from this text.\n    ', '\n    Graph node index: 1\n    Page display name (metadata field page_name, not the prediction target): U.S. Consulate General Mumbai\n    Facebook numeric id (metadata): 191483281412\n    Facebook page in the MU SocialAds / SNAP graph; page_type label is omitted from this text.\n    ']


In [8]:
import torch
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

# ✅ FIX: Use `documents` from Cell 7, not a nonexistent df
text_embeddings = encoder.encode(documents, show_progress_bar=True, batch_size=64)
text_embeddings = np.array(text_embeddings)  

print("Text embedding shape:", text_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/352 [00:00<?, ?it/s]

Text embedding shape: (22470, 384)


In [9]:
import torch

node_to_idx = {node: i for i, node in enumerate(nodes)}
edge_index = []

for _, row in edges.iterrows():
    src_id = int(row["id_1"])
    dst_id = int(row["id_2"])
    # ✅ FIX: skip edges whose nodes aren't in our node set
    if src_id in node_to_idx and dst_id in node_to_idx:
        src = node_to_idx[src_id]
        dst = node_to_idx[dst_id]
        edge_index.append([src, dst])
        edge_index.append([dst, src])

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
print("Edge index shape:", edge_index.shape)

Edge index shape: torch.Size([2, 342004])


In [10]:
from torch_geometric.data import Data
import numpy as np

X_combined = np.concatenate([X, text_embeddings], axis=1)

x = torch.tensor(X_combined, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

print(data)

Data(x=[22470, 5098], edge_index=[2, 342004], y=[22470])


In [11]:
from sklearn.model_selection import train_test_split
import torch

indices = list(range(len(Y)))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=Y   # ✅ CRITICAL FIX
)

train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask

In [12]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

# Two SAGE layers (same pattern as gnn-only-facebook GraphSAGE; no separate Linear head)
class GNN_RAG_Model(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

In [13]:
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = int(np.max(Y)) + 1
model = GNN_RAG_Model(
    in_channels=data.num_features,
    hidden_channels=128,
    num_classes=num_classes,
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=50  
)

In [14]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score

OUTPUT_DIR = "weights"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_acc = 0  # track best

def train():
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(101):
    loss = train()
    scheduler.step(loss)

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
            y_te = data.y[data.test_mask].cpu().numpy()
            proba = torch.softmax(out[data.test_mask], dim=1).cpu().numpy()
            p_te = pred[data.test_mask].cpu().numpy()
            f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
            c = proba.shape[1]
            try:
                if c == 2:
                    aucv = roc_auc_score(y_te, proba[:, 1])
                else:
                    aucv = roc_auc_score(
                        y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
                    )
            except ValueError:
                aucv = float("nan")
            print(
                f"Epoch {epoch:4d} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} "
                f"| Val Acc: {acc:.4f} | Val F1: {f1v:.4f} | Val AUC: {aucv:.4f}"
            )

            # Save best checkpoint (mirrors train_facebook.py's best.pth logic)
            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pth")



Epoch    0 | Loss: 1.3913 | LR: 0.010000 | Val Acc: 0.5901 | Val F1: 0.4922 | Val AUC: 0.9349
Epoch   10 | Loss: 0.2349 | LR: 0.010000 | Val Acc: 0.9221 | Val F1: 0.9172 | Val AUC: 0.9870
Epoch   20 | Loss: 0.1555 | LR: 0.010000 | Val Acc: 0.9408 | Val F1: 0.9379 | Val AUC: 0.9897
Epoch   30 | Loss: 0.1259 | LR: 0.010000 | Val Acc: 0.9435 | Val F1: 0.9403 | Val AUC: 0.9909
Epoch   40 | Loss: 0.1089 | LR: 0.010000 | Val Acc: 0.9430 | Val F1: 0.9402 | Val AUC: 0.9919
Epoch   50 | Loss: 0.0974 | LR: 0.010000 | Val Acc: 0.9466 | Val F1: 0.9437 | Val AUC: 0.9922
Epoch   60 | Loss: 0.0788 | LR: 0.010000 | Val Acc: 0.9457 | Val F1: 0.9427 | Val AUC: 0.9924
Epoch   70 | Loss: 0.0861 | LR: 0.010000 | Val Acc: 0.9446 | Val F1: 0.9408 | Val AUC: 0.9925
Epoch   80 | Loss: 0.0631 | LR: 0.010000 | Val Acc: 0.9473 | Val F1: 0.9443 | Val AUC: 0.9926
Epoch   90 | Loss: 0.0570 | LR: 0.010000 | Val Acc: 0.9493 | Val F1: 0.9468 | Val AUC: 0.9928
Epoch  100 | Loss: 0.0598 | LR: 0.010000 | Val Acc: 0.9464 |

In [15]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def test():
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
        acc = int(correct) / int(data.test_mask.sum())
        y_te = data.y[data.test_mask].cpu().numpy()
        proba = torch.softmax(out[data.test_mask], dim=1).cpu().numpy()
        p_te = pred[data.test_mask].cpu().numpy()
        f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
        c = proba.shape[1]
        try:
            if c == 2:
                aucv = roc_auc_score(y_te, proba[:, 1])
            else:
                aucv = roc_auc_score(
                    y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
                )
        except ValueError:
            aucv = float("nan")
    return acc, f1v, aucv

acc, f1v, aucv = test()
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1v:.4f} | AUC (macro OVR): {aucv:.4f}")


Test Accuracy: 0.9464 | F1 (macro): 0.9433 | AUC (macro OVR): 0.9929


In [16]:
# -- Load best checkpoint and save final weights -----
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pth"))
torch.save(model.state_dict(), f"{OUTPUT_DIR}/model_weights_facebook.pth")
print("Saved model_weights_facebook.pth")

# Embeddings = conv1 + ReLU (conv2 outputs class logits; 2-layer GNN)
model.eval()
with torch.no_grad():
    x_in = data.x
    ei   = data.edge_index
    embeddings = F.relu(model.conv1(x_in, ei))  # [num_nodes, hidden_channels]

np.save(f"{OUTPUT_DIR}/embeddings_facebook.npy", embeddings.cpu().numpy())
print("Saved embeddings_facebook.npy")

print("Training successfully completed!")

from IPython.display import FileLink
display(FileLink(f"{OUTPUT_DIR}/model_weights_facebook.pth"))
display(FileLink(f"{OUTPUT_DIR}/embeddings_facebook.npy"))

Saved model_weights_facebook.pth
Saved embeddings_facebook.npy
Training successfully completed!


/kaggle/working/weights/model_weights_facebook.pth

/kaggle/working/weights/embeddings_facebook.npy

In [17]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def accuracy(pred, y):
    return (pred == y).sum().item() / len(y)

def f1_auc_mask(mask):
    y_ = data.y[mask].cpu().numpy()
    proba = torch.softmax(out[mask], dim=1).cpu().numpy()
    p_ = pred[mask].cpu().numpy()
    f1_ = f1_score(y_, p_, average="macro", zero_division=0)
    c_ = proba.shape[1]
    try:
        if c_ == 2:
            auc_ = roc_auc_score(y_, proba[:, 1])
        else:
            auc_ = roc_auc_score(
                y_, proba, multi_class="ovr", average="macro", labels=np.arange(c_)
            )
    except ValueError:
        auc_ = float("nan")
    return f1_, auc_

model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

train_acc = accuracy(pred[data.train_mask], data.y[data.train_mask])
test_acc = accuracy(pred[data.test_mask], data.y[data.test_mask])
f1_tr, auc_tr = f1_auc_mask(data.train_mask)
f1_te, auc_te = f1_auc_mask(data.test_mask)

print(f"Train Acc: {train_acc:.4f} | F1: {f1_tr:.4f} | AUC: {auc_tr:.4f}")
print(f"Test Acc: {test_acc:.4f} | F1: {f1_te:.4f} | AUC: {auc_te:.4f}")


Train Acc: 0.9872 | F1: 0.9872 | AUC: 0.9997
Test Acc: 0.9493 | F1: 0.9468 | AUC: 0.9928
